### YOLOv8 Segmentação



#### imports e configuração global

In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from pathlib import Path
import urllib.request
import os
import time
import warnings
warnings.filterwarnings("ignore")

# Ultralytics / YOLOv8
from ultralytics import YOLO

# InsightFace
import insightface
from insightface.app import FaceAnalysis
from insightface.data import get_image as ins_get_image

# Configuração de device
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
import ultralytics
print(f"Versao do Yolo: {ultralytics.__version__}")
print(f"Versao do InsightFace: {insightface.__version__}")



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/gustavoaranha/PycharmProjects/mba-ia/05-visao-computacional/.venv/lib/python3.11/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/gustavoaranha/PycharmProjects/mba-ia/05-visao-computacional/.venv/lib/python3.11/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/Users/gustavoaranha/PycharmProjects/mba

Device: cpu
Versao do Yolo: 8.4.29
Versao do InsightFace: 0.7.3


In [ ]:
print("Carregando YOLOv8n (segmentacao)...")
yolo_seg = YOLO("yolov8n-seg.pt")

print(f"   Classe 'person' = ID {[k for k, v in yolo_seg.names.items() if v == 'person'][0]}")

print(60*'-')
print(f"Classes disponíveis:")
for id, name in yolo_seg.names.items():
    print(f"   ID {id}: {name}")
    
print(60*'-')
print(f"   Total de classes: {len(yolo_seg.names)}")


In [ ]:
img_bgr = cv2.imread("data/t1.jpg")
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

h, w = img_rgb.shape[:2]
results = yolo_seg(img_rgb)
r = results[0]

person_class_id = 0
final_mask = np.zeros((h, w), dtype=np.uint8)

if r.masks is not None and r.boxes is not None:
    classes = r.boxes.cls.cpu().numpy().astype(int)
    masks = r.masks.data.cpu().numpy()  # shape: [N, Hm, Wm]

    for cls_id, mask in zip(classes, masks):
        if cls_id == person_class_id:
            mask_bin = (mask > 0.5).astype(np.uint8) * 255
            mask_resized = cv2.resize(mask_bin, (w, h), interpolation=cv2.INTER_NEAREST)
            final_mask = np.maximum(final_mask, mask_resized)

result_rgb = np.zeros_like(img_rgb)
result_rgb[final_mask > 0] = img_rgb[final_mask > 0]

plt.figure(figsize=(12, 6))

plt.subplot(1, 3, 1)
plt.imshow(img_rgb)
plt.title("Imagem original")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(final_mask, cmap="gray")
plt.title("Máscara das pessoas")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(result_rgb)
plt.title("Pessoas com fundo preto")
plt.axis("off")

plt.tight_layout()
plt.show()

#### câmera ao vivo



In [ ]:
def obtem_pessoas(img_rgb, yolo_seg):
    h, w = img_rgb.shape[:2]
    results = yolo_seg(img_rgb)
    r = results[0]
    person_class_id = 0
    final_mask = np.zeros((h, w), dtype=np.uint8)
    if r.masks is not None and r.boxes is not None:
        classes = r.boxes.cls.cpu().numpy().astype(int)
        masks = r.masks.data.cpu().numpy()  # shape: [N, Hm, Wm]

        for cls_id, mask in zip(classes, masks):
            if cls_id == person_class_id:
                mask_bin = (mask > 0.5).astype(np.uint8) * 255
                mask_resized = cv2.resize(mask_bin, (w, h), interpolation=cv2.INTER_NEAREST)
                final_mask = np.maximum(final_mask, mask_resized)

    result_rgb = np.zeros_like(img_rgb)
    result_rgb[final_mask > 0] = img_rgb[final_mask > 0]
    return result_rgb


In [ ]:
#  Câmera local (descomente para usar localmente) 
# use apenas localmente!!!
from IPython.display import display, clear_output

id_camera = 0
cap = cv2.VideoCapture(id_camera)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 900)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 600)
print("Pressione 'q' para encerrar | 'r' para registrar identidade")
using_cv = False

frame_idx = 0
captura_por_frames = 5
while True:
    ret, frame = cap.read()
    if not ret: break
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    if using_cv:
        try:
            cv2.imshow("YOLOv8 + InsightFace", frame)
            key = cv2.waitKey(1) & 0xFF
            if key == ord('q'): break
        except Exception as e:
            using_cv = False
            
    frame = obtem_pessoas(frame, yolo_seg)
    if not using_cv:
        # 3. Transforma o array em imagem PIL
        img = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        # time.sleep(0.05)
        clear_output(wait=True)
        display(img)    
        
    frame_idx += 1
    if frame_idx >100: break

cap.release()

if using_cv:
    try:
        cv2.destroyAllWindows()
        cv2.waitKey(1)
    except Exception as e:
        pass

print("Câmera encerrada.")